# Config and Secrets Management

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/06-config-and-secrets.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic fastapi pydantic pydantic-settings uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

A production AI service has at least a dozen configuration values: the model name, max tokens, temperature, timeout, retry count, rate limit, service port, log level, and the API key. Where do these values live?

Most teams end up with a combination of hardcoded constants, environment variable reads scattered across six files, a config YAML that only works in staging, and an `.env` file that nobody can quite remember the correct format of. When the production service silently uses a default model because a staging environment variable was not set in production, the degradation is invisible until a user complains that responses are wrong. When a misconfigured timeout causes every request to h...

## The Concept

### The 3-Tier Config Resolution Order

Every well-structured service has config that flows from three sources, with each tier able to override the tier below it.

```
Tier 3 (highest priority): Environment Variables
    ANTHROPIC_API_KEY=sk-ant-...
    LOG_LEVEL=debug

          overrides
              |
              v

Tier 2 (medium priority): Config File (YAML or TOML)
    model: claude-3-5-haiku-20241022
    max_tokens: 1024
    timeout_seconds: 30

          overrides
              |
              v

Tier 1 (lowest priority): Defaults in Code
    model = "claude-3-5-haiku-20241022"
    ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Config and Secrets Management for a FastAPI AI Service.

Demonstrates the 3-tier config resolution pattern using pydantic-settings:
  1. Defaults in code (lowest priority)
  2. .env file (if present)
  3. Environment variables (highest priority)

Secrets (ANTHROPIC_API_KEY) are required with no defaults.
Non-secret config has typed defaults and validation constraints.

Usage:
    # Fail-fast demo (missing required secret):
    python main.py

    # Normal startup:
    ANTHROPIC_API_KEY=sk-... uvicorn main:app --reload

    # Override a default:
    ANTHROPIC_API_KEY=sk-... MAX_TOKENS=512 uvicorn main:app --reload
"""

import sys

import anthropic
from fastapi import Depends, FastAPI, HTTPException
from pydantic import ValidationError
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field
from functools import lru_cache

### Settings definition

In [ ]:
class Settings(BaseSettings):
    """
    Typed configuration for the AI service.

    Resolution order (highest wins):
      1. Environment variables
      2. .env file (only if it exists; missing .env is not an error)
      3. Default values below

    Validation runs at instantiation (startup). A misconfigured service fails
    immediately with a clear error rather than silently using wrong values.
    """

    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        case_sensitive=False,
    )

    # SECRETS: no defaults; must be provided via environment or .env file
    anthropic_api_key: str = Field(..., description="Anthropic API key (required)")

    # MODEL CONFIG: non-secret, safe to commit or bake into Docker ENV
    model: str = Field(
        default="claude-3-5-haiku-20241022",
        description="Claude model ID",
    )
    max_tokens: int = Field(
        default=1024,
        ge=1,
        le=8192,
        description="Max tokens in model response",
    )
    temperature: float = Field(
        default=0.0,
        ge=0.0,
        le=1.0,
        description="Sampling temperature",
    )

    # SERVICE CONFIG
    port: int = Field(default=8000, ge=1024, le=65535)
    log_level: str = Field(default="info")
    timeout_seconds: int = Field(default=30, ge=1, le=300)
    max_retries: int = Field(default=3, ge=0, le=10)


@lru_cache(maxsize=1)
def get_settings() -> Settings:
    """
    Load and validate settings once, cache the result.

    The lru_cache ensures environment variables are read and validated
    exactly once per process. Subsequent calls return the cached object.
    """
    return Settings()

### Fail-fast startup validation

In [ ]:
# Validate config before binding a port or accepting connections.
# The service exits immediately with a clear error if required values are missing
# or if any value fails its type/range constraints.
try:
    _startup_settings = get_settings()
    print(f"Config loaded: model={_startup_settings.model}, "
          f"max_tokens={_startup_settings.max_tokens}, "
          f"port={_startup_settings.port}")
except ValidationError as e:
    print(f"[STARTUP ERROR] Configuration is invalid. Service will not start.\n{e}",
          file=sys.stderr)
    sys.exit(1)

### Application

In [ ]:
app = FastAPI(title="AI Service", version="1.0")


def get_client(settings: Settings = Depends(get_settings)) -> anthropic.Anthropic:
    """Dependency that creates the Anthropic client from validated settings."""
    return anthropic.Anthropic(api_key=settings.anthropic_api_key)


@app.get("/health")
def health(settings: Settings = Depends(get_settings)):
    """Health check: returns 200 with current config summary (no secrets)."""
    return {
        "status": "ok",
        "model": settings.model,
        "max_tokens": settings.max_tokens,
        "log_level": settings.log_level,
    }


@app.get("/config")
def config(settings: Settings = Depends(get_settings)):
    """
    Returns non-secret config for debugging.
    Never expose secrets here -- only safe-to-log values.
    """
    return {
        "model": settings.model,
        "max_tokens": settings.max_tokens,
        "temperature": settings.temperature,
        "timeout_seconds": settings.timeout_seconds,
        "max_retries": settings.max_retries,
        "log_level": settings.log_level,
        # anthropic_api_key is intentionally omitted
    }


@app.post("/generate")
def generate(
    prompt: str,
    settings: Settings = Depends(get_settings),
    client: anthropic.Anthropic = Depends(get_client),
):
    """
    Generate a response using settings loaded from the environment.

    All config values come from the validated Settings object.
    No os.environ.get() calls in business logic.
    """
    if not prompt.strip():
        raise HTTPException(status_code=400, detail="prompt must not be empty")

    msg = client.messages.create(
        model=settings.model,
        max_tokens=settings.max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "text": msg.content[0].text,
        "model": msg.model,
        "input_tokens": msg.usage.input_tokens,
        "output_tokens": msg.usage.output_tokens,
    }


# ---------------------------------------------------------------------------
# Dev entrypoint
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import uvicorn
uvicorn.run("main:app", host="0.0.0.0", port=_startup_settings.port, reload=True)

---

## Self-Check

Answer these without running code first:

**1. Which design decision would have made this misconfiguration visible immediately at deployment time?**

- A. Add a comment in the code explaining what MODEL controls.
- B. Validate all config at startup using a typed Settings class with fail-fast behavior so the service exits if required values differ from expectations.
- C. Log every request's model name in the response JSON so users can see which model answered.
- D. Pin the model name as a hardcoded constant so environment variables cannot override it.

**2. What happens when the service starts?**

- A. The service starts and uses max_tokens=0 because the string 'unlimited' cannot be converted to int, so pydantic falls back to the default.
- B. The service raises a ValidationError at startup and exits with a non-zero code, printing a clear error identifying the MAX_TOKENS field.
- C. The service starts with max_tokens='unlimited' as a string and crashes on the first request that calls the API with it.
- D. pydantic-settings ignores environment variables that cannot be coerced to the declared type and uses the default instead.

**3. What is the immediate action to take after discovering this, and what process change prevents it recurring?**

- A. Delete the .env file from the repository and force-push. Then add .env to .gitignore. The key is still valid because GitHub only checks for keys in the current HEAD.
- B. Rotate the API key immediately (assume it is compromised), then add .env to .gitignore and create a .env.example with placeholder values.
- C. Add a note in the README saying developers should not use real keys in .env files.
- D. Move the .env file to a subdirectory that git does not track by default.

**4. What is the behavior of the service, assuming fail-fast startup validation is implemented?**

- A. The pod starts, enters the Running state, and crashes only on the first request that uses the API key.
- B. The pod fails at startup with a ValidationError, never enters Running state, and Kubernetes reports CrashLoopBackOff.
- C. pydantic-settings treats the missing field as None and the service starts with api_key=None.
- D. Kubernetes injects a default empty string for any missing environment variable, so the service starts with api_key=''.

**5. Which value does settings.model have when the service starts?**

- A. claude-opus-4-5 (the .env file takes precedence over shell environment variables)
- B. claude-3-5-haiku-20241022 (shell environment variables take precedence over the .env file)
- C. The service raises a conflict error because the same field is defined in two places.
- D. The behavior is undefined and depends on which file was loaded first.

**6. What is the concrete problem with mixing hardcoded constants and raw os.environ.get() calls instead of a unified typed Settings class?**

- A. No problem. This approach is simpler and achieves the same security properties.
- B. Hardcoded constants cannot be overridden for A/B testing or environment-specific tuning without a code change and redeployment; os.environ.get() calls are untyped and unvalidated, so a wrong DATABASE_URL format only fails at connection time.
- C. The problem is performance: os.environ.get() is slower than reading a Python constant.
- D. os.environ.get() is not available in all Python environments and should be replaced with pathlib.

**7. Which secret management approach is appropriate at this scale?**

- A. Store all three keys in environment variables. Rotation requires a pod restart, which is acceptable.
- B. Move to AWS Secrets Manager or HashiCorp Vault, which support dynamic secret rotation and provide an audit trail of every read. The service fetches secrets at startup (and optionally refreshes them).
- C. Store the keys in a YAML config file that is mounted as a Kubernetes ConfigMap so the team can edit it without redeploying.
- D. Store the keys in a .env file committed to a private repository. Only give developers with senior engineer title access to clone it.

_Answers are in `checks.json` in the lesson directory._